# Strategic Classification: Cluster Shift Poisoning

Non-backdoor poisoning adapted from `black_box_cluster`. Shifts epsilon fraction of
each class's **strategic features only** toward the opposing class mean, confusing the
decision boundary. Labels are never modified.

Compares clean vs poisoned for both RGD and PerfGD (logreg + linear performativity).

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import sys
import os

cwd = os.getcwd()  # poison-perf/strategic/
root = os.path.dirname(cwd)  # poison-perf/
sys.path.append(root)

from utils.algorithms import RGD, PerfGD
from utils.strategic_setup import setup_strategic_classification
from data_prep import load_data
from response import strategic_cluster_shift

## Load Data

In [ ]:
data_dir = "data/GiveMeSomeCredit"
data_file = "cs-training.csv"
data_path = os.path.join(root, data_dir, data_file)
X_np, Y_np, data = load_data(data_path)

n_samples, d_plus_bias = X_np.shape
print("n_samples =", n_samples, ", d+1 (with bias) =", d_plus_bias)

strat_features = np.array([1, 6, 8]) - 1  # 0-indexed
print("\nStrategic features:")
for i, feature in enumerate(strat_features):
    print(f"  {i}: [{feature}] {data.columns[feature + 1]}")

device = torch.device("cpu")
X_full = torch.from_numpy(X_np).float().to(device)
Y_full = torch.from_numpy(Y_np).float().to(device)

lam = 1.0 / n_samples
alpha_strat = 1.0

## Configuration

In [ ]:
n = n_samples
max_iter = 100

# Smoothness-based learning rate
smoothness = (X_full ** 2).sum().item() / (4.0 * n_samples)
eta = 2.0 / (smoothness + 2.0 * lam)
print(f"smoothness = {smoothness:.4f}, eta = {eta:.4f}")

proj_theta = lambda t: t

# Poisoning parameters
poison_epsilon = 0.3   # fraction of each class to poison
poison_delta = 1.0     # shift magnitude

print(f"poison_epsilon = {poison_epsilon}")
print(f"poison_delta = {poison_delta}")

## Setup Experiment (LogReg + Linear)

In [ ]:
mu_f, sigma, D_theta, loss, theta_0, grad2_est, f_hat = setup_strategic_classification(
    X_base=X_full,
    Y_base=Y_full,
    strat_features=strat_features.tolist(),
    alpha=alpha_strat,
    lam=lam,
    classifier="logreg",
    performativity="linear",
    perfGD=True,
)

print(f"theta_0 shape: {theta_0.shape} ({theta_0.numel()} params)")

## Visualize Poisoning Effect

Show the strategic feature distributions before and after cluster shift poisoning.

In [ ]:
# Get one batch of strategically-shifted data
z_clean = D_theta(theta_0, n)
z_poisoned = strategic_cluster_shift(
    z_clean, theta_0,
    strat_features=strat_features.tolist(),
    epsilon=poison_epsilon,
    delta=poison_delta,
)

strat_names = ["RevolvingUtilization", "OpenCreditLines", "RealEstateLoans"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, (ax, feat_idx, name) in enumerate(zip(axes, strat_features, strat_names)):
    y = z_clean[-1, :].numpy()
    x_clean_feat = z_clean[feat_idx, :].numpy()
    x_poison_feat = z_poisoned[feat_idx, :].numpy()

    # Clean
    ax.hist(x_clean_feat[y == 0], bins=40, alpha=0.4, color="tab:blue", label="Clean y=0", density=True)
    ax.hist(x_clean_feat[y == 1], bins=40, alpha=0.4, color="tab:red", label="Clean y=1", density=True)
    # Poisoned
    ax.hist(x_poison_feat[y == 0], bins=40, alpha=0.3, color="blue", label="Poisoned y=0", density=True, histtype="step", linewidth=2)
    ax.hist(x_poison_feat[y == 1], bins=40, alpha=0.3, color="red", label="Poisoned y=1", density=True, histtype="step", linewidth=2)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("Feature value")
    if i == 0:
        ax.legend(fontsize=7)

plt.suptitle(f"Cluster Shift Poisoning (eps={poison_epsilon}, delta={poison_delta})", fontsize=12)
plt.tight_layout()
plt.show()

## RGD: Clean vs Poisoned

In [ ]:
# Clean RGD
theta_rgd_clean, all_theta_rgd_clean, all_losses_rgd_clean = RGD(
    D_theta=D_theta, loss=loss, theta_0=theta_0.clone(),
    n=n, eta=eta, max_iter=max_iter, proj_theta=proj_theta, return_losses=True,
)

# Poisoned RGD
theta_rgd_poison, all_theta_rgd_poison, all_losses_rgd_poison = RGD(
    D_theta=D_theta, loss=loss, theta_0=theta_0.clone(),
    n=n, eta=eta, max_iter=max_iter, proj_theta=proj_theta, return_losses=True,
    poison_function=strategic_cluster_shift,
    strat_features=strat_features.tolist(),
    epsilon=poison_epsilon,
    delta=poison_delta,
)

print(f"RGD Clean final loss:    {all_losses_rgd_clean[-1]:.4f}")
print(f"RGD Poisoned final loss: {all_losses_rgd_poison[-1]:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(all_losses_rgd_clean, marker="o", markersize=3, label="Clean", color="tab:green")
ax1.plot(all_losses_rgd_poison, marker="o", markersize=3, label="Poisoned", color="tab:red")
ax1.set_xlabel("Iteration")
ax1.set_ylabel("Performative loss")
ax1.set_title("RGD Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

clean_norms = [t.norm().item() for t in all_theta_rgd_clean]
poison_norms = [t.norm().item() for t in all_theta_rgd_poison]
ax2.plot(clean_norms, marker="o", markersize=3, label="Clean", color="tab:green")
ax2.plot(poison_norms, marker="o", markersize=3, label="Poisoned", color="tab:red")
ax2.set_xlabel("Iteration")
ax2.set_ylabel(r"$\|\theta\|_2$")
ax2.set_title(r"RGD $\|\theta\|_2$")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## PerfGD: Clean vs Poisoned

In [ ]:
# Clean PerfGD
theta_pgd_clean, all_theta_pgd_clean, all_losses_pgd_clean = PerfGD(
    f_hat=f_hat, grad2_est=grad2_est, D_theta=D_theta, loss=loss,
    theta_0=theta_0.clone(), n=n, eta=eta, max_iter=max_iter,
    proj_theta=proj_theta, return_losses=True,
)

# Poisoned PerfGD
theta_pgd_poison, all_theta_pgd_poison, all_losses_pgd_poison = PerfGD(
    f_hat=f_hat, grad2_est=grad2_est, D_theta=D_theta, loss=loss,
    theta_0=theta_0.clone(), n=n, eta=eta, max_iter=max_iter,
    proj_theta=proj_theta, return_losses=True,
    poison_function=strategic_cluster_shift,
    strat_features=strat_features.tolist(),
    epsilon=poison_epsilon,
    delta=poison_delta,
)

print(f"PerfGD Clean final loss:    {all_losses_pgd_clean[-1]:.4f}")
print(f"PerfGD Poisoned final loss: {all_losses_pgd_poison[-1]:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(all_losses_pgd_clean, marker="o", markersize=3, label="Clean", color="tab:green")
ax1.plot(all_losses_pgd_poison, marker="o", markersize=3, label="Poisoned", color="tab:red")
ax1.set_xlabel("Iteration")
ax1.set_ylabel("Performative loss")
ax1.set_title("PerfGD Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

clean_norms = [t.norm().item() for t in all_theta_pgd_clean]
poison_norms = [t.norm().item() for t in all_theta_pgd_poison]
ax2.plot(clean_norms, marker="o", markersize=3, label="Clean", color="tab:green")
ax2.plot(poison_norms, marker="o", markersize=3, label="Poisoned", color="tab:red")
ax2.set_xlabel("Iteration")
ax2.set_ylabel(r"$\|\theta\|_2$")
ax2.set_title(r"PerfGD $\|\theta\|_2$")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## RGD vs PerfGD Comparison

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(all_losses_rgd_clean, marker="o", markersize=3, label="RGD Clean", color="tab:green")
ax1.plot(all_losses_rgd_poison, marker="o", markersize=3, label="RGD Poisoned", color="tab:red")
ax1.plot(all_losses_pgd_clean, marker="s", markersize=3, label="PerfGD Clean", color="tab:green", linestyle="--")
ax1.plot(all_losses_pgd_poison, marker="s", markersize=3, label="PerfGD Poisoned", color="tab:red", linestyle="--")
ax1.set_xlabel("Iteration")
ax1.set_ylabel("Performative loss")
ax1.set_title("All Runs: Loss")
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# Theta difference: how far did poisoning move the final theta?
rgd_diff = [torch.norm(c - p).item() for c, p in zip(all_theta_rgd_clean, all_theta_rgd_poison)]
pgd_diff = [torch.norm(c - p).item() for c, p in zip(all_theta_pgd_clean, all_theta_pgd_poison)]
ax2.plot(rgd_diff, marker="o", markersize=3, label="RGD", color="tab:blue")
ax2.plot(pgd_diff, marker="s", markersize=3, label="PerfGD", color="tab:orange")
ax2.set_xlabel("Iteration")
ax2.set_ylabel(r"$\|\theta_{clean} - \theta_{poison}\|_2$")
ax2.set_title("Parameter Divergence (Clean vs Poisoned)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

In [ ]:
print(f"{'':>20} {'Final Loss':>12} {'||th||_2':>12} {'Delta Loss':>12}")
print("-" * 60)
print(f"{'RGD Clean':<20} {all_losses_rgd_clean[-1]:>12.4f} {all_theta_rgd_clean[-1].norm().item():>12.4f}")
print(f"{'RGD Poisoned':<20} {all_losses_rgd_poison[-1]:>12.4f} {all_theta_rgd_poison[-1].norm().item():>12.4f} {all_losses_rgd_poison[-1] - all_losses_rgd_clean[-1]:>+12.4f}")
print(f"{'PerfGD Clean':<20} {all_losses_pgd_clean[-1]:>12.4f} {all_theta_pgd_clean[-1].norm().item():>12.4f}")
print(f"{'PerfGD Poisoned':<20} {all_losses_pgd_poison[-1]:>12.4f} {all_theta_pgd_poison[-1].norm().item():>12.4f} {all_losses_pgd_poison[-1] - all_losses_pgd_clean[-1]:>+12.4f}")

print(f"\nFinal theta divergence:")
print(f"  RGD:   ||theta_clean - theta_poison||_2 = {torch.norm(theta_rgd_clean - theta_rgd_poison).item():.4f}")
print(f"  PerfGD: ||theta_clean - theta_poison||_2 = {torch.norm(theta_pgd_clean - theta_pgd_poison).item():.4f}")

## Delta Sweep

Sweep over poison strength (delta) to see how poisoning impact scales.

In [ ]:
delta_vals = np.linspace(0.0, 3.0, 13)
rgd_final_losses = []
pgd_final_losses = []
rgd_theta_divs = []
pgd_theta_divs = []

# Clean baselines (already computed)
rgd_clean_loss = all_losses_rgd_clean[-1]
pgd_clean_loss = all_losses_pgd_clean[-1]

for d_val in delta_vals:
    print(f"delta = {d_val:.2f}", end=" ")

    # RGD
    th_r, all_th_r, losses_r = RGD(
        D_theta=D_theta, loss=loss, theta_0=theta_0.clone(),
        n=n, eta=eta, max_iter=max_iter, proj_theta=proj_theta, return_losses=True,
        poison_function=strategic_cluster_shift,
        strat_features=strat_features.tolist(),
        epsilon=poison_epsilon, delta=d_val,
    )
    rgd_final_losses.append(losses_r[-1])
    rgd_theta_divs.append(torch.norm(theta_rgd_clean - th_r).item())

    # PerfGD
    th_p, all_th_p, losses_p = PerfGD(
        f_hat=f_hat, grad2_est=grad2_est, D_theta=D_theta, loss=loss,
        theta_0=theta_0.clone(), n=n, eta=eta, max_iter=max_iter,
        proj_theta=proj_theta, return_losses=True,
        poison_function=strategic_cluster_shift,
        strat_features=strat_features.tolist(),
        epsilon=poison_epsilon, delta=d_val,
    )
    pgd_final_losses.append(losses_p[-1])
    pgd_theta_divs.append(torch.norm(theta_pgd_clean - th_p).item())

    print(f"-> RGD loss={losses_r[-1]:.4f}, PerfGD loss={losses_p[-1]:.4f}")

print("Done.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(delta_vals, rgd_final_losses, marker="o", label="RGD", color="tab:blue")
ax1.plot(delta_vals, pgd_final_losses, marker="s", label="PerfGD", color="tab:orange")
ax1.axhline(rgd_clean_loss, color="tab:blue", linestyle="--", alpha=0.5, label="RGD clean")
ax1.axhline(pgd_clean_loss, color="tab:orange", linestyle="--", alpha=0.5, label="PerfGD clean")
ax1.set_xlabel(r"$\delta$ (shift magnitude)")
ax1.set_ylabel("Final performative loss")
ax1.set_title(f"Loss vs Poison Strength (eps={poison_epsilon})")
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

ax2.plot(delta_vals, rgd_theta_divs, marker="o", label="RGD", color="tab:blue")
ax2.plot(delta_vals, pgd_theta_divs, marker="s", label="PerfGD", color="tab:orange")
ax2.set_xlabel(r"$\delta$ (shift magnitude)")
ax2.set_ylabel(r"$\|\theta_{clean} - \theta_{poison}\|_2$")
ax2.set_title(f"Parameter Divergence vs Poison Strength (eps={poison_epsilon})")
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()